# 01 — Ingestão e Unificação das Bases

**Objetivo:** Carregar as duas bases fictícias (`input_wide_ficticio` e `temporal_output_ficticio`),
validar integridade, unificá-las em uma base analítica única e salvar o resultado.

**Referência SPEC:** Seções 1-3 (Objetivo, Definição do Target, Estrutura da Base)

---

| Entrada | Descrição |
|---|---|
| `input_wide_ficticio.parquet` | Features mensalizadas (1 linha = 1 colaborador × 1 mês) |
| `temporal_output_ficticio.parquet` | Target de turnover voluntário + metadados temporais |

| Saída | Descrição |
|---|---|
| `data/gold/base_analitica.parquet` | Base unificada pronta para construção do dataset |

## 1 · Setup & Configuração

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"pandas {pd.__version__}  ·  numpy {np.__version__}")
print(f"Execução: {datetime.now():%Y-%m-%d %H:%M}")

pandas 2.3.3  ·  numpy 2.3.5
Execução: 2026-04-27 18:04


In [2]:
# ════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ════════════════════════════════════════════════════════════════
PROJECT_ROOT = Path.cwd().parent          # raiz do repo
DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_GOLD    = PROJECT_ROOT / "data" / "gold"

# Parâmetros do target
JANELA_PREDICAO    = 4          # meses à frente para definir turnover
COL_TARGET         = f"fg_status_vol_{JANELA_PREDICAO}m"
COL_ID             = "id_colaborador"
COL_DATA           = "dt_mes_ano"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_RAW     : {DATA_RAW}")
print(f"DATA_GOLD    : {DATA_GOLD}")
print(f"Target       : {COL_TARGET}")

PROJECT_ROOT : G:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo
DATA_RAW     : G:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\raw
DATA_GOLD    : G:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold
Target       : fg_status_vol_4m


## 2 · Criação da Estrutura de Pastas

In [3]:
# Estrutura de pastas do projeto (criada automaticamente)
DIRS = {
    "data/raw":                  "Bases brutas de entrada",
    "data/gold":                 "Base analítica unificada",
    "data/processed":            "Datasets de treino/teste/validação",
    "data/processed/splits":     "Splits treino/val/teste por grupo",
    "models":                    "Modelos serializados (.joblib)",
    "reports":                   "Relatórios e métricas",
    "reports/figures":           "Gráficos e visualizações",
    "reports/feature_selection": "Resultados de feature selection",
    "reports/shap":              "SHAP values e plots",
}

for subdir, desc in DIRS.items():
    p = PROJECT_ROOT / subdir
    p.mkdir(parents=True, exist_ok=True)

print("✓ Estrutura de pastas criada:")
for subdir, desc in DIRS.items():
    print(f"  {subdir:<35s} — {desc}")

✓ Estrutura de pastas criada:
  data/raw                            — Bases brutas de entrada
  data/gold                           — Base analítica unificada
  data/processed                      — Datasets de treino/teste/validação
  data/processed/splits               — Splits treino/val/teste por grupo
  models                              — Modelos serializados (.joblib)
  reports                             — Relatórios e métricas
  reports/figures                     — Gráficos e visualizações
  reports/feature_selection           — Resultados de feature selection
  reports/shap                        — SHAP values e plots


## 3 · Carga das Bases

In [4]:
# ── Carregar input_wide (features) ────────────────────────────
wide = pd.read_parquet(DATA_RAW / "input_wide_ficticio.parquet")
print(f"input_wide_ficticio : {wide.shape[0]:,} linhas × {wide.shape[1]} colunas")
print(f"  IDs únicos: {wide[COL_ID].nunique():,}")
print(f"  Período   : {wide[COL_DATA].min()} → {wide[COL_DATA].max()}")

# ── Carregar temporal_output (target) ─────────────────────────
output = pd.read_parquet(DATA_RAW / "temporal_output_ficticio.parquet")
print(f"\ntemporal_output_ficticio : {output.shape[0]:,} linhas × {output.shape[1]} colunas")
print(f"  IDs únicos: {output[COL_ID].nunique():,}")
print(f"  Período   : {output[COL_DATA].min()} → {output[COL_DATA].max()}")

input_wide_ficticio : 202,519 linhas × 117 colunas
  IDs únicos: 14,174
  Período   : 2024-01-01 00:00:00 → 2025-12-01 00:00:00

temporal_output_ficticio : 207,435 linhas × 12 colunas
  IDs únicos: 14,722
  Período   : 2024-01-01 00:00:00 → 2025-12-01 00:00:00


In [5]:
# ── Carregar mensalizada (população completa) ─────────────────
# Usada para calcular indicadores de contexto por grupo (cargo / uorg / gestor)
# sem depender do subconjunto amostrado do wide
mensalizada = pd.read_parquet(DATA_RAW / "mensalizada_ficticio.parquet")
mensalizada[COL_DATA] = pd.to_datetime(mensalizada[COL_DATA])

print(f"mensalizada_ficticio : {mensalizada.shape[0]:,} linhas × {mensalizada.shape[1]} colunas")
print(f"  IDs únicos: {mensalizada[COL_ID].nunique():,}")
print(f"  Período   : {mensalizada[COL_DATA].min():%Y-%m} → {mensalizada[COL_DATA].max():%Y-%m}")
print(f"  Meses     : {mensalizada[COL_DATA].nunique()}")


mensalizada_ficticio : 207,435 linhas × 63 colunas
  IDs únicos: 14,722
  Período   : 2024-01 → 2025-12
  Meses     : 24


In [6]:
# ── Resumo das colunas do wide ────────────────────────────────
print("Colunas do input_wide:")
for i, c in enumerate(wide.columns, 1):
    dtype = wide[c].dtype
    n_null = wide[c].isna().sum()
    print(f"  {i:2d}. {c:<45s} {str(dtype):<12s} nulls={n_null}")

Colunas do input_wide:
   1. id_colaborador                                object       nulls=0
   2. dt_mes_ano                                    datetime64[ns] nulls=0
   3. vl_adc_not_dias_med_3m                        object       nulls=0
   4. vl_adc_not_horas_med_3m                       object       nulls=0
   5. vl_atv_3m_perc_equipe_med_3m                  object       nulls=0
   6. vl_atv_3m_perc_equipe_med_ult                 object       nulls=0
   7. vl_atv_3m_perc_uorg_med_3m                    object       nulls=0
   8. vl_atv_3m_perc_uorg_med_ult                   object       nulls=0
   9. vl_atv_6m_perc_equipe_med_3m                  object       nulls=0
  10. vl_atv_6m_perc_equipe_med_ult                 object       nulls=0
  11. vl_atv_6m_perc_uorg_med_3m                    object       nulls=0
  12. vl_atv_6m_perc_uorg_med_ult                   object       nulls=0
  13. vl_atraso_dias_med_3m_x                       object       nulls=0
  14. vl_atraso_horas_med_

  36. vl_dias_sabado_med_3m_x                       object       nulls=0
  37. vl_dias_noturno_med_3m_x                      object       nulls=0


  38. vl_falta_dias_med_3m_x                        object       nulls=0
  39. vl_falta_horas_med_3m_x                       object       nulls=0
  40. vl_horas_mes_med_3m                           object       nulls=0
  41. vl_cid_infecto_dias_sum_3m                    object       nulls=0
  42. Insalubridade_nzero_3m                        object       nulls=0
  43. vl_banco_horas_pagamento_med_3m               object       nulls=0
  44. vl_diurno_perc_med_3m                         object       nulls=0
  45. vl_rv_max_3m                                  object       nulls=0
  46. vl_rv_med_3m                                  object       nulls=0
  47. vl_rv_meses_nao_zero_3m                       object       nulls=0
  48. vl_rv_ult                                     object       nulls=0
  49. Salário substituição_nzero_3m                 object       nulls=0
  50. Sobreaviso_nzero_3m                           object       nulls=0
  51. vl_headcount_equipe_med_3m                   

  97. vl_horas_extras_extraordinarias_horas_med_3m  float64      nulls=0


  98. vl_horas_extras_extraordinarias_horas_med_6m  float64      nulls=0
  99. vl_adicional_noturno_dias_med_3m              float64      nulls=0
  100. vl_adicional_noturno_dias_med_6m              float64      nulls=0
  101. vl_adicional_noturno_horas_med_3m             float64      nulls=0
  102. vl_adicional_noturno_horas_med_6m             float64      nulls=0
  103. vl_atraso_dias_med_3m_y                       float64      nulls=0
  104. vl_atraso_dias_med_6m                         float64      nulls=0
  105. vl_atraso_horas_med_3m_y                      float64      nulls=0
  106. vl_atraso_horas_med_6m                        float64      nulls=0
  107. vl_falta_dias_med_3m_y                        float64      nulls=0
  108. vl_falta_dias_med_6m                          float64      nulls=0
  109. vl_falta_horas_med_3m_y                       float64      nulls=0
  110. vl_falta_horas_med_6m                         float64      nulls=0
  111. vl_economico_taxa_desemprego    

In [7]:
# ── Resumo das colunas do output ──────────────────────────────
print("Colunas do temporal_output:")
for i, c in enumerate(output.columns, 1):
    dtype = output[c].dtype
    n_null = output[c].isna().sum()
    print(f"  {i:2d}. {c:<45s} {str(dtype):<12s} nulls={n_null}")

Colunas do temporal_output:
   1. id_colaborador                                object       nulls=0
   2. dt_mes_ano                                    datetime64[us] nulls=0
   3. fg_status_vol_3m                              float64      nulls=43030
   4. fg_status_vol_4m                              float64      nulls=79359
   5. fg_status_vol_5m                              float64      nulls=66265
   6. fg_status_vol_6m                              float64      nulls=76991
   7. fg_status_vol_7m                              float64      nulls=87257
   8. fg_status_vol_8m                              float64      nulls=97049
   9. fg_status_vol_9m                              float64      nulls=106459
  10. fg_status_vol_10m                             float64      nulls=115492
  11. fg_status_vol_11m                             float64      nulls=124241
  12. fg_status_vol_12m                             float64      nulls=132723


## 4 · Validação Pré-Merge

In [8]:
# ── Normalizar data para datetime em ambas as bases ───────────
wide[COL_DATA]   = pd.to_datetime(wide[COL_DATA])
output[COL_DATA] = pd.to_datetime(output[COL_DATA])

# ── Garantir tipo string no ID ────────────────────────────────
wide[COL_ID]   = wide[COL_ID].astype(str)
output[COL_ID] = output[COL_ID].astype(str)

# ── Interseção de IDs ─────────────────────────────────────────
ids_wide   = set(wide[COL_ID].unique())
ids_output = set(output[COL_ID].unique())
ids_common = ids_wide & ids_output

print(f"IDs no wide   : {len(ids_wide):,}")
print(f"IDs no output : {len(ids_output):,}")
print(f"IDs comuns    : {len(ids_common):,}")
print(f"Só no wide    : {len(ids_wide - ids_output):,}")
print(f"Só no output  : {len(ids_output - ids_wide):,}")

# ── Interseção de períodos ────────────────────────────────────
meses_wide   = set(wide[COL_DATA].unique())
meses_output = set(output[COL_DATA].unique())
meses_common = meses_wide & meses_output
print(f"\nMeses no wide   : {len(meses_wide)}")
print(f"Meses no output : {len(meses_output)}")
print(f"Meses comuns    : {len(meses_common)}")

IDs no wide   : 14,174
IDs no output : 14,722
IDs comuns    : 14,174
Só no wide    : 0
Só no output  : 548

Meses no wide   : 24
Meses no output : 24
Meses comuns    : 24


In [9]:
# ── Checar duplicatas nas chaves de merge ─────────────────────
dup_wide   = wide.duplicated(subset=[COL_ID, COL_DATA]).sum()
dup_output = output.duplicated(subset=[COL_ID, COL_DATA]).sum()

print(f"Duplicatas (id, mês) no wide   : {dup_wide}")
print(f"Duplicatas (id, mês) no output : {dup_output}")

if dup_wide > 0:
    print("⚠ Removendo duplicatas do wide (mantendo última ocorrência)")
    wide = wide.drop_duplicates(subset=[COL_ID, COL_DATA], keep="last")

if dup_output > 0:
    print("⚠ Removendo duplicatas do output (mantendo última ocorrência)")
    output = output.drop_duplicates(subset=[COL_ID, COL_DATA], keep="last")

Duplicatas (id, mês) no wide   : 0
Duplicatas (id, mês) no output : 15
⚠ Removendo duplicatas do output (mantendo última ocorrência)


## 5 · Merge — Unificação

In [10]:
# ── Identificar colunas do output que não estão no wide ──────
# Evitar duplicação de colunas comuns (exceto as chaves de merge)
merge_keys = [COL_ID, COL_DATA]
cols_output_only = [c for c in output.columns if c not in wide.columns or c in merge_keys]

print(f"Colunas do output a incorporar: {len(cols_output_only) - len(merge_keys)}")
for c in cols_output_only:
    if c not in merge_keys:
        print(f"  + {c}")

Colunas do output a incorporar: 10
  + fg_status_vol_3m
  + fg_status_vol_4m
  + fg_status_vol_5m
  + fg_status_vol_6m
  + fg_status_vol_7m
  + fg_status_vol_8m
  + fg_status_vol_9m
  + fg_status_vol_10m
  + fg_status_vol_11m
  + fg_status_vol_12m


In [11]:
# ── Inner join: só pares (id, mês) presentes em ambas ────────
df = wide.merge(
    output[cols_output_only],
    on=merge_keys,
    how="inner"
)

n_lost_wide   = len(wide) - len(df)
n_lost_output = len(output) - len(df)

print(f"Resultado do merge (inner):")
print(f"  wide          : {len(wide):>10,} linhas")
print(f"  output        : {len(output):>10,} linhas")
print(f"  base unificada: {len(df):>10,} linhas")
print(f"  perdidas wide : {n_lost_wide:>10,}")
print(f"  perdidas output: {n_lost_output:>10,}")
print(f"\n  IDs únicos    : {df[COL_ID].nunique():,}")
print(f"  Colunas       : {df.shape[1]}")

Resultado do merge (inner):
  wide          :    202,519 linhas
  output        :    207,420 linhas
  base unificada:    202,519 linhas
  perdidas wide :          0
  perdidas output:      4,901

  IDs únicos    : 14,174
  Colunas       : 127


## 5.5 · Indicadores Contextuais por Grupo (SPEC §3.4)

Para variáveis categóricas com alta cardinalidade (`ds_cargo`, `ds_uorg`, `ds_gestor`)
uma simples codificação one-hot geraria centenas de colunas esparsas e introduziria
risco de multicolinearidade. A alternativa adotada aqui é **quantificá-las por comportamento**:
calculamos, para cada grupo × mês, três indicadores baseados na população completa da mensalizada:

| Indicador | Definição |
|---|---|
| **Headcount** | Média do headcount mensal (count de `fg_ativo == 1`) nos **4 meses** (atual + 3 anteriores) |
| **Turnover voluntário** | Soma de `fg_dem_vol2 == 1` / soma de `fg_ativo == 1` nos mesmos 4 meses |
| **Turnover involuntário** | Soma de `fg_dem_inv2 == 1` / soma de `fg_ativo == 1` nos mesmos 4 meses |

A janela de 4 meses é calculada de forma **rolling** (sem lookahead), garantindo que
para qualquer registro em `dt_mes_ano = T`, apenas dados de `T`, `T-1`, `T-2`, `T-3`
sejam usados — mesmo que o grupo não exista em algum mês anterior (`min_periods=1`).

> **Por que isso funciona bem para modelos preditivos?** Captura dynamics de retenção de
> equipes/departamentos/gestores ao longo do tempo, sem expor informação futura ao modelo —
> um erro clássico em datasets de RH.


In [12]:
# ════════════════════════════════════════════════════════════════
# INDICADORES CONTEXTUAIS POR GRUPO
# Janela deslizante de 4 meses sobre a população completa da mensalizada
# para quantificar variáveis categóricas de alta cardinalidade.
# ════════════════════════════════════════════════════════════════

JANELA_ROLL  = 4                     # mês atual + 3 anteriores
GRUPOS_ROLL  = [                     # (coluna, sufixo nos nomes das features)
    ("ds_cargo",  "cargo"),
    ("ds_uorg",   "uorg"),
    ("ds_gestor", "gestor"),
]
COL_ATIVO    = "fg_ativo"
COL_DEM_VOL  = "fg_dem_vol"
COL_DEM_INV  = "fg_dem_inv"


def calcular_indicadores_grupo(mens_df, group_col, group_suffix, col_data, n_meses=4):
    """
    Calcula headcount, turnover voluntário e involuntário com janela rolling
    de N meses por categoria, sem vazamento de dados futuros.

    Parâmetros
    ----------
    mens_df      : Mensalizada completa (toda a população, não apenas a amostra)
    group_col    : Coluna de agrupamento (ex.: 'ds_cargo')
    group_suffix : Sufixo para nomear as colunas de saída (ex.: 'cargo')
    col_data     : Nome da coluna de data (dt_mes_ano)
    n_meses      : Tamanho da janela rolling (padrão: 4)

    Retorna
    -------
    DataFrame com: [col_data, group_col, hc_Xm, tv_vol_Xm, tv_inv_Xm]
    """
    # Verificar colunas necessárias
    for c in [col_data, group_col, COL_ATIVO, COL_DEM_VOL, COL_DEM_INV]:
        if c not in mens_df.columns:
            raise ValueError(f"Coluna '{c}' não encontrada na mensalizada")

    # ── Passo 1: contagens mensais por grupo ─────────────────────
    monthly = (
        mens_df
        .groupby([col_data, group_col], observed=True, sort=True)
        .agg(
            cnt_ativo=(COL_ATIVO,   lambda x: (x == 1).sum()),
            cnt_vol  =(COL_DEM_VOL, lambda x: (x == 1).sum()),
            cnt_inv  =(COL_DEM_INV, lambda x: (x == 1).sum()),
        )
        .reset_index()
        .sort_values([group_col, col_data])
    )

    # ── Passo 2: rolling de N meses por grupo ────────────────────
    grp = monthly.groupby(group_col, sort=False)
    monthly["hc_roll"]   = grp["cnt_ativo"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).mean()
    )
    monthly["sum_ativo"] = grp["cnt_ativo"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).sum()
    )
    monthly["sum_vol"]   = grp["cnt_vol"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).sum()
    )
    monthly["sum_inv"]   = grp["cnt_inv"].transform(
        lambda x: x.rolling(n_meses, min_periods=1).sum()
    )

    # ── Passo 3: taxas de turnover (sum_ativo como denominador) ──
    denom = monthly["sum_ativo"].replace(0, np.nan)
    monthly[f"vl_hc_{group_suffix}_{n_meses}m"]     = monthly["hc_roll"]
    monthly[f"vl_tv_vol_{group_suffix}_{n_meses}m"] = monthly["sum_vol"] / denom
    monthly[f"vl_tv_inv_{group_suffix}_{n_meses}m"] = monthly["sum_inv"] / denom

    feat_cols = [
        f"vl_hc_{group_suffix}_{n_meses}m",
        f"vl_tv_vol_{group_suffix}_{n_meses}m",
        f"vl_tv_inv_{group_suffix}_{n_meses}m",
    ]
    return monthly[[col_data, group_col] + feat_cols]


print("Função calcular_indicadores_grupo definida ✓")

Função calcular_indicadores_grupo definida ✓


In [13]:
# ── Aplicar para cada variável de grupo ───────────────────────
stats_cols_added = []

for group_col, group_suffix in GRUPOS_ROLL:
    if group_col not in df.columns:
        print(f"⚠ {group_col} não encontrada no df — pulando")
        continue
    if group_col not in mensalizada.columns:
        print(f"⚠ {group_col} não encontrada na mensalizada — pulando")
        continue

    stats = calcular_indicadores_grupo(
        mensalizada, group_col, group_suffix, COL_DATA, JANELA_ROLL
    )

    new_cols = [c for c in stats.columns if c not in [COL_DATA, group_col]]
    n_antes  = len(df)
    df = df.merge(stats, on=[COL_DATA, group_col], how="left")

    # Sanity: inner merge não deve alterar número de linhas
    assert len(df) == n_antes, f"Merge alterou contagem! {n_antes} → {len(df)}"

    stats_cols_added.extend(new_cols)
    n_grupos = stats[group_col].nunique()
    for c in new_cols:
        null_pct = df[c].isna().mean() * 100
        print(f"  [{group_suffix}] {c}: média={df[c].mean():.4f} | nulos={null_pct:.1f}%")
    print(f"  ↳ {n_grupos} grupos '{group_col}' | {len(stats)} pares (mês × grupo)")

print(f"\n✓ {len(stats_cols_added)} colunas de contexto adicionadas:")
print(f"  {stats_cols_added}")
print(f"\nBase df após indicadores: {df.shape[0]:,} linhas × {df.shape[1]} colunas")


  [cargo] vl_hc_cargo_4m: média=552.7342 | nulos=0.0%
  [cargo] vl_tv_vol_cargo_4m: média=0.0382 | nulos=0.0%
  [cargo] vl_tv_inv_cargo_4m: média=0.0138 | nulos=0.0%
  ↳ 119 grupos 'ds_cargo' | 2557 pares (mês × grupo)


  [uorg] vl_hc_uorg_4m: média=212.2669 | nulos=0.0%
  [uorg] vl_tv_vol_uorg_4m: média=0.0383 | nulos=0.0%
  [uorg] vl_tv_inv_uorg_4m: média=0.0141 | nulos=0.0%
  ↳ 91 grupos 'ds_uorg' | 1935 pares (mês × grupo)


  [gestor] vl_hc_gestor_4m: média=16.3754 | nulos=0.0%
  [gestor] vl_tv_vol_gestor_4m: média=0.0391 | nulos=0.2%
  [gestor] vl_tv_inv_gestor_4m: média=0.0104 | nulos=0.2%
  ↳ 1811 grupos 'ds_gestor' | 22271 pares (mês × grupo)

✓ 9 colunas de contexto adicionadas:
  ['vl_hc_cargo_4m', 'vl_tv_vol_cargo_4m', 'vl_tv_inv_cargo_4m', 'vl_hc_uorg_4m', 'vl_tv_vol_uorg_4m', 'vl_tv_inv_uorg_4m', 'vl_hc_gestor_4m', 'vl_tv_vol_gestor_4m', 'vl_tv_inv_gestor_4m']

Base df após indicadores: 202,519 linhas × 136 colunas


In [14]:

# ── 5.6 · Grupo Operacional (ds_grupo) ───────────────────────
# 'ds_grupo' identifica o grupo operacional do colaborador:
# Vendas / Transporte / Fábrica.
# A coluna está na mensalizada (long), mas não no wide (pivot de features).
# Fazemos um left-join por [id_colaborador, dt_mes_ano] para trazê-la
# para o df principal antes de salvar a base_analitica.
#
# Nota: bases fictícias antigas usavam "Área 1/2/3" — o replace() abaixo
# faz o mapeamento quando necessário mas preserva os valores já corretos
# ("Vendas", "Transporte", "Fábrica") sem convertê-los em NaN.

# Remover TODAS as variantes da coluna (re-execução idempotente)
cols_grupo_drop = [c for c in df.columns if c.startswith("ds_grupo")]
if cols_grupo_drop:
    df = df.drop(columns=cols_grupo_drop)

grupo_lookup = (
    mensalizada[[COL_ID, COL_DATA, "ds_grupo"]]
    .drop_duplicates(subset=[COL_ID, COL_DATA])
)

n_antes = len(df)
df = df.merge(grupo_lookup, on=[COL_ID, COL_DATA], how="left")
assert len(df) == n_antes, "Merge ds_grupo alterou o número de linhas!"

# Normalizar nomes antigos ("Área X") para nomes do pipeline
# replace() mantém o valor original para chaves fora do mapeamento
_grp_map = {"Área 1": "Transporte", "Área 2": "Vendas", "Área 3": "Fábrica"}
df["ds_grupo"] = df["ds_grupo"].replace(_grp_map)

pct_null = df["ds_grupo"].isna().mean() * 100
print(f"ds_grupo adicionado ao df")
print(f"  Nulos : {df['ds_grupo'].isna().sum():,}  ({pct_null:.1f}%)")
print(f"\nDistribuição por grupo:")
print(df["ds_grupo"].value_counts(dropna=False).to_string())


ds_grupo adicionado ao df
  Nulos : 0  (0.0%)

Distribuição por grupo:
ds_grupo
Transporte    104905
Vendas         72008
Fábrica        25606


## 6 · Validação do Target

In [15]:
# ── Verificar presença da coluna de target ────────────────────
target_cols = [c for c in df.columns if "fg_status_vol" in c.lower()]
print(f"Colunas de target encontradas: {target_cols}")

if COL_TARGET not in df.columns:
    # Tentar encontrar a coluna mais próxima
    candidates = [c for c in df.columns if "fg_status_vol" in c]
    if candidates:
        COL_TARGET = candidates[0]
        print(f"⚠ Target ajustado para: {COL_TARGET}")
    else:
        raise ValueError(f"Coluna de target não encontrada. Colunas disponíveis: {list(df.columns)}")

print(f"\nDistribuição do target ({COL_TARGET}):")
print(df[COL_TARGET].value_counts(dropna=False).to_string())
print(f"\nTaxa de turnover: {df[COL_TARGET].mean():.2%}")

Colunas de target encontradas: ['fg_status_vol_3m', 'fg_status_vol_4m', 'fg_status_vol_5m', 'fg_status_vol_6m', 'fg_status_vol_7m', 'fg_status_vol_8m', 'fg_status_vol_9m', 'fg_status_vol_10m', 'fg_status_vol_11m', 'fg_status_vol_12m']

Distribuição do target (fg_status_vol_4m):
fg_status_vol_4m
0.0    110160
NaN     74506
1.0     17853

Taxa de turnover: 13.95%


In [16]:

# ── fg_sem_output: identifica meses sem janela de predição válida ──
# Meses Set/2025+ não têm 4 meses de forward window dentro da base.
# Esses registros NÃO devem ser removidos pelo dropna — serão usados
# como base de aplicação (score sem target).
CUTOFF_SEM_OUTPUT = pd.Timestamp("2025-09-01")

df["fg_sem_output"] = (df[COL_DATA] >= CUTOFF_SEM_OUTPUT).astype(int)

n_com_output = (df["fg_sem_output"] == 0).sum()
n_sem_output = (df["fg_sem_output"] == 1).sum()

print(f"fg_sem_output aplicado:")
print(f"  fg_sem_output = 0 (com output):  {n_com_output:,} registros  ({df[COL_DATA][df['fg_sem_output']==0].min():%Y-%m} → {df[COL_DATA][df['fg_sem_output']==0].max():%Y-%m})")
print(f"  fg_sem_output = 1 (sem output):  {n_sem_output:,} registros  (Set/2025+)")
print(f"\nDistribuição do target ({COL_TARGET}) nos registros COM output (fg_sem_output=0):")
mask_com = df["fg_sem_output"] == 0
print(df.loc[mask_com, COL_TARGET].value_counts(dropna=False).to_string())
print(f"\nTaxa de turnover (registros com output, sem NaN de afastamento):")
target_com = df.loc[mask_com, COL_TARGET]
taxa = target_com.mean()
print(f"  {taxa:.2%}  (inclui NaN de afastamento/deslig. involuntário)")


fg_sem_output aplicado:
  fg_sem_output = 0 (com output):  166,098 registros  (2024-01 → 2025-08)
  fg_sem_output = 1 (sem output):  36,421 registros  (Set/2025+)

Distribuição do target (fg_status_vol_4m) nos registros COM output (fg_sem_output=0):
fg_status_vol_4m
0.0    110160
NaN     38494
1.0     17444

Taxa de turnover (registros com output, sem NaN de afastamento):
  13.67%  (inclui NaN de afastamento/deslig. involuntário)


## 7 · Validação de Qualidade

In [17]:
# ── Nulos por coluna ──────────────────────────────────────────
null_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
null_pct_pos = null_pct[null_pct > 0]

if len(null_pct_pos) > 0:
    print(f"Colunas com nulos ({len(null_pct_pos)}):")
    for col, pct in null_pct_pos.items():
        print(f"  {col:<45s} {pct:5.1f}%  ({df[col].isna().sum():,} linhas)")
else:
    print("✓ Nenhuma coluna com nulos")

print(f"\nTotal de nulos na base: {df.isna().sum().sum():,}")

Colunas com nulos (16):
  fg_status_vol_12m                              63.1%  (127,849 linhas)
  fg_status_vol_11m                              59.0%  (119,386 linhas)
  fg_status_vol_10m                              54.6%  (110,638 linhas)
  fg_status_vol_9m                               50.2%  (101,608 linhas)
  fg_status_vol_8m                               45.5%  (92,202 linhas)
  fg_status_vol_7m                               40.7%  (82,416 linhas)
  fg_status_vol_4m                               36.8%  (74,506 linhas)
  fg_status_vol_6m                               35.6%  (72,154 linhas)
  fg_status_vol_5m                               30.3%  (61,431 linhas)
  fg_status_vol_3m                               18.9%  (38,203 linhas)
  vl_tv_inv_gestor_4m                             0.2%  (419 linhas)
  vl_tv_vol_gestor_4m                             0.2%  (419 linhas)
  vl_tv_vol_cargo_4m                              0.0%  (85 linhas)
  vl_tv_inv_cargo_4m                          


Total de nulos na base: 881,403


In [18]:
# ── Tipos de dados ────────────────────────────────────────────
print("Resumo de tipos:")
print(df.dtypes.value_counts().to_string())

# ── Registros por mês ─────────────────────────────────────────
print(f"\nRegistros por mês:")
print(df.groupby(COL_DATA)[COL_ID].nunique().to_string())

Resumo de tipos:
object            76
float64           46
int64             15
datetime64[ns]     1

Registros por mês:
dt_mes_ano
2024-01-01    7857
2024-02-01    7759
2024-03-01    7734
2024-04-01    7713
2024-05-01    7774
2024-06-01    7777
2024-07-01    7913
2024-08-01    7993
2024-09-01    8203
2024-10-01    8506
2024-11-01    8798
2024-12-01    9007
2025-01-01    8630
2025-02-01    8587
2025-03-01    8562
2025-04-01    8563
2025-05-01    8581
2025-06-01    8682
2025-07-01    8709
2025-08-01    8750
2025-09-01    8934
2025-10-01    9024
2025-11-01    9141
2025-12-01    9322


In [19]:
# ── Taxa de turnover por grupo (se existir) ──────────────────
col_grupo = None
for c in ["grupo", "ds_grupo", "agrupamento_final"]:
    if c in df.columns:
        col_grupo = c
        break

if col_grupo:
    print(f"Taxa de turnover por {col_grupo}:")
    stats = df.groupby(col_grupo).agg(
        n_linhas=(COL_ID, "count"),
        n_ids=(COL_ID, "nunique"),
        taxa_turnover=(COL_TARGET, "mean"),
    )
    stats["taxa_turnover"] = stats["taxa_turnover"].map("{:.2%}".format)
    print(stats.to_string())
else:
    print("Coluna de grupo não encontrada — análise por grupo não disponível")

Taxa de turnover por ds_grupo:
            n_linhas  n_ids taxa_turnover
ds_grupo                                 
Fábrica        25606   1771        14.95%
Transporte    104905   7423        13.47%
Vendas         72008   5210        14.29%


## 8 · Salvar Base Analítica

In [20]:

# ════════════════════════════════════════════════════════════════
# SALVAR TRÊS BASES  —  SPEC §0.6
#
#  base_tt_raw  : treino + teste  (Jan/2024–Mai/2025, target válido)
#  base_val_raw : validação       (Jun/2025–Ago/2025, target válido)
#  base_apl_raw : aplicação       (Set/2025+, fg_sem_output=1, sem target)
#
# Regras:
#   - tt e val: fg_sem_output==0, depois dropna(target)
#   - tt: dt_mes_ano <= Mai/2025
#   - val: Jun/2025 <= dt_mes_ano <= Ago/2025
#   - apl: fg_sem_output==1  (TODOS os registros, sem dropna)
# ════════════════════════════════════════════════════════════════

CUTOFF_TT  = pd.Timestamp("2025-05-31")   # último mês de treino/teste
CUTOFF_VAL_START = pd.Timestamp("2025-06-01")
CUTOFF_VAL_END   = pd.Timestamp("2025-08-31")

# ── base_tt_raw ───────────────────────────────────────────────
base_tt = (
    df[(df["fg_sem_output"] == 0) & (df[COL_DATA] <= CUTOFF_TT)]
    .dropna(subset=[COL_TARGET])
    .copy()
    .reset_index(drop=True)
)
out_tt = DATA_GOLD / "base_tt_raw.parquet"
base_tt.to_parquet(out_tt, index=False)

# ── base_val_raw ──────────────────────────────────────────────
base_val = (
    df[
        (df["fg_sem_output"] == 0) &
        (df[COL_DATA] >= CUTOFF_VAL_START) &
        (df[COL_DATA] <= CUTOFF_VAL_END)
    ]
    .dropna(subset=[COL_TARGET])
    .copy()
    .reset_index(drop=True)
)
out_val = DATA_GOLD / "base_val_raw.parquet"
base_val.to_parquet(out_val, index=False)

# ── base_apl_raw ──────────────────────────────────────────────
base_apl = (
    df[df["fg_sem_output"] == 1]
    .copy()
    .reset_index(drop=True)
)
out_apl = DATA_GOLD / "base_apl_raw.parquet"
base_apl.to_parquet(out_apl, index=False)

# ── Reportar ──────────────────────────────────────────────────
print(f"{'Base':<18} {'Shape':>16}  {'IDs':>8}  {'Período':>20}  {'Turnover':>10}")
print("-" * 80)
for nome, base, out_path in [
    ("base_tt_raw",  base_tt,  out_tt),
    ("base_val_raw", base_val, out_val),
    ("base_apl_raw", base_apl, out_apl),
]:
    n_ids  = base[COL_ID].nunique()
    per_st = base[COL_DATA].min().strftime("%Y-%m") if len(base) > 0 else "—"
    per_en = base[COL_DATA].max().strftime("%Y-%m") if len(base) > 0 else "—"
    taxa_s = f"{base[COL_TARGET].mean():.2%}" if nome != "base_apl_raw" else "NaN (sem target)"
    mb = out_path.stat().st_size / 1e6
    print(f"  {nome:<16} {str(base.shape):>16}  {n_ids:>8,}  {per_st} → {per_en}  {taxa_s:>14}  ({mb:.1f} MB)")
    print(f"  ✓ Salvo em: {out_path}")


Base                          Shape       IDs               Período    Turnover
--------------------------------------------------------------------------------
  base_tt_raw         (110503, 138)    10,676  2024-01 → 2025-05          13.68%  (35.6 MB)
  ✓ Salvo em: G:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_tt_raw.parquet
  base_val_raw         (17101, 138)     6,097  2025-06 → 2025-08          13.60%  (6.1 MB)
  ✓ Salvo em: G:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_val_raw.parquet
  base_apl_raw         (36421, 138)     9,979  2025-09 → 2025-12  NaN (sem target)  (12.4 MB)
  ✓ Salvo em: G:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_apl_raw.parquet


In [21]:

# ── Resumo final ──────────────────────────────────────────────
print("="*70)
print("RESUMO — INGESTÃO")
print("="*70)
print(f"  Base unificada (antes do split) : {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  IDs únicos                      : {df[COL_ID].nunique():,}")
print(f"  Período completo                : {df[COL_DATA].min():%Y-%m} → {df[COL_DATA].max():%Y-%m}")
print()
print(f"  {'Base':<18} {'Linhas':>10}  {'IDs':>8}  {'Período':>20}  {'Turnover':>12}")
print("  " + "-"*72)
for nome, base in [
    ("base_tt_raw",  base_tt),
    ("base_val_raw", base_val),
    ("base_apl_raw", base_apl),
]:
    n_ids  = base[COL_ID].nunique()
    per_st = base[COL_DATA].min().strftime("%Y-%m") if len(base) > 0 else "—"
    per_en = base[COL_DATA].max().strftime("%Y-%m") if len(base) > 0 else "—"
    if nome != "base_apl_raw":
        taxa_s = f"{base[COL_TARGET].mean():.2%}"
    else:
        taxa_s = "NaN(sem target)"
    print(f"  {nome:<18} {len(base):>10,}  {n_ids:>8,}  {per_st} → {per_en}  {taxa_s:>14}")

print()
if "ds_grupo" in df.columns:
    print("  Taxa de turnover por grupo (base_tt_raw):")
    grupo_stats = base_tt.groupby("ds_grupo").agg(
        n_linhas=(COL_ID, "count"),
        n_ids=(COL_ID, "nunique"),
        taxa_turnover=(COL_TARGET, "mean"),
    )
    for g, row in grupo_stats.iterrows():
        print(f"    {g:<14}: {row['n_ids']:>4,} ids  |  {row['n_linhas']:>6,} reg  |  {row['taxa_turnover']:.2%}")
print("="*70)


RESUMO — INGESTÃO
  Base unificada (antes do split) : 202,519 linhas × 138 colunas
  IDs únicos                      : 14,174
  Período completo                : 2024-01 → 2025-12

  Base                   Linhas       IDs               Período      Turnover
  ------------------------------------------------------------------------
  base_tt_raw           110,503    10,676  2024-01 → 2025-05          13.68%
  base_val_raw           17,101     6,097  2025-06 → 2025-08          13.60%
  base_apl_raw           36,421     9,979  2025-09 → 2025-12  NaN(sem target)

  Taxa de turnover por grupo (base_tt_raw):
    Fábrica       : 1,348.0 ids  |  13,907.0 reg  |  14.52%
    Transporte    : 5,623.0 ids  |  57,381.0 reg  |  13.28%
    Vendas        : 3,818.0 ids  |  39,215.0 reg  |  13.98%
